# Squamate Vertebra Shape Completion with Uncertainty Estimation and Evaluation
---
This notebook demonstrates the base NSM shape completion workflow, plus ad hoc uncertainty estimation.

## 1. Imports
---
Import modules, add a monkey patch, and define plotting helper functions.

In [ ]:
# System
from datetime import datetime
import itertools
import os
import random

# Meshes
import pyvista as pv
pv.set_jupyter_backend('static')
import pymskt.mesh.meshes as meshes

# Data and ML
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

# NSM
from NSM.helper_funcs import decode_sdf, load_config, load_model_and_latents
import NSM.uncertainty as uncert # Uncertainty estimation

# Monkey patch for data types ----
from NSM.helper_funcs import safe_load_mesh_scalars, fixed_point_coords
meshes.Mesh.load_mesh_scalars = safe_load_mesh_scalars
meshes.Mesh.point_coords = property(fixed_point_coords)
import pymskt.mesh.meshTools as meshTools
_original_signed_distance_to_mesh = meshTools.pcu.signed_distance_to_mesh
def _signed_distance_to_mesh_patch(pts, points, faces):
    pts = np.asarray(pts, dtype=np.float64)     # force double precision
    points = np.asarray(points, dtype=np.float64)
    faces = np.asarray(faces, dtype=np.int32)   # ensure integer type for faces
    return _original_signed_distance_to_mesh(pts, points, faces)
meshTools.pcu.signed_distance_to_mesh = _signed_distance_to_mesh_patch
# End monkey patch ----

In [ ]:
# --- Plotting helper functions ---
def preview_meshes(mesh_list, jupyter_backend='static'):
    """Plot a list of meshes in a row."""
    n_meshes = len(mesh_list)
    pl = pv.Plotter(shape=(1,n_meshes), window_size=[400*n_meshes,400])
    pl.link_views()
    for i, mesh in enumerate(mesh_list):
        pl.subplot(0,i)
        pl.add_mesh(mesh)
    pl.show(jupyter_backend=jupyter_backend)

def render_and_save_uncertainty(input_mesh, recon_mesh_unc, whole_mesh, 
                                scalar_name,
                                mesh_name, wholemesh_basename, 
                                out_name, jupyter_backend='static'):
    """
    Plot input partial mesh, reconstructed mesh with uncertainty, and original whole mesh (if provided)
    in a single figure. Save png and gif.
    """
    # PyVista Plotter
    pl = pv.Plotter(shape=(1,3), window_size=[1200,400])
    pl.link_views()
    # 1) Input mesh
    pl.subplot(0,0)
    pl.add_mesh(uncert.unit_scale_mesh(input_mesh), color="yellow")
    pl.add_title(f"Input fragmented mesh\n{mesh_name}", font_size=6, font='times')
    # 2) Reconstructed mesh with uncertainty
    pl.subplot(0,1)
    scalar_bar_args = dict(title=scalar_name, title_font_size=14, label_font_size=14, font_family='times')
    pl.add_mesh(uncert.unit_scale_mesh(recon_mesh_unc), scalar_bar_args=scalar_bar_args, scalars=recon_mesh_unc[scalar_name])
    pl.add_title(f"Reconstructed mesh with uncertainty\n{mesh_name}", font_size=6, font='times')
    # 3) Original whole mesh
    if whole_mesh:
        pl.subplot(0,2)
        pl.add_mesh(uncert.unit_scale_mesh(whole_mesh), color="orange")
        pl.add_title(f"Original whole mesh\n{wholemesh_basename}", font_size=6, font='times')
    
    # Plot and save
    img_path = out_name + ".png"
    pl.show(jupyter_backend=jupyter_backend, screenshot=img_path)
    print(f"Saved PNG to {img_path}")

    # Generate GIF and save
    gif_path = out_name + ".gif"
    pl.open_gif(gif_path, fps=5)
    n_frames = 60
    for i in range(n_frames):
        pl.camera.Azimuth(720 / n_frames)
        pl.camera.elevation = -30 + 30*np.sin(2*np.pi * i/n_frames)
        pl.write_frame()
    pl.close()
    print(f"Saved GIF to {gif_path}")

## 2. Specify model, mesh, and output paths

In [ ]:
# --- Input paths ---
# Base directory, run name (trained model version), and mesh directories
DATA_BASE_DIR = "./"
DATA_RUN_NAME = "run_v57"
DATA_MESH_DIR = os.path.join(DATA_BASE_DIR, DATA_RUN_NAME, "shape_completion/meshes/partial_meshes_remove4/partial_meshes") # Directory containing input partial meshes
if whole_mesh_available := True: # Change to False if ground truth whole mesh is unavailable (i.e., at test time)
    DATA_WHOLEMESH_DIR = os.path.join(DATA_BASE_DIR, DATA_RUN_NAME, "shape_completion/meshes/alignedModels") # Directory containing original whole meshes
else:
    DATA_WHOLEMESH_DIR = None
# Paths to model config, model, and latent codes files
CKPT = "3000" # Checkpoint to use
MODEL_PATH = os.path.join(DATA_BASE_DIR, DATA_RUN_NAME, f"model/{CKPT}.pth")
LC_PATH = os.path.join(DATA_BASE_DIR, DATA_RUN_NAME, f"latent_codes/{CKPT}.pth")
CONFIG_PATH = os.path.join(DATA_BASE_DIR, DATA_RUN_NAME, "model_params_config.json")

# --- Output path ---
# User output directory
OUTPUT_BASE_DIR = f"./output/{DATA_RUN_NAME}/week15/uncertainty"
os.makedirs(OUTPUT_BASE_DIR, exist_ok=True)

print(f"Model path: {MODEL_PATH}")
print(f"Latent codes path: {LC_PATH}")
print(f"Config path: {CONFIG_PATH}")
print(f"Input mesh directory: {DATA_MESH_DIR}")
if whole_mesh_available:
    print(f"Original whole mesh directory: {DATA_WHOLEMESH_DIR}")
print(f"Output directory: {OUTPUT_BASE_DIR}")

## 3. Load autodecoder model and input mesh


### Load config, model, and latent codes

In [ ]:
# --- Load model config, model, and latent codes ---
config = load_config(CONFIG_PATH)
device = config.get("device", "cuda:0" if torch.cuda.is_available() else "cpu")
model, _, latent_codes = load_model_and_latents(MODEL_PATH, LC_PATH, config, device)

print(f"Using device: {device}")
print(f"Loaded model at checkpoint: {CKPT}")
print(f"Latent dimensions: {config['latent_size']}")
print(f"Number of latent codes: {len(latent_codes)}")

### Load an input partial mesh (and corresponding ground truth whole mesh if available)

In [ ]:
# --- Load random partial mesh ---
mesh_basename = random.choice(os.listdir(DATA_MESH_DIR))
print(f"Selected input partial mesh: {mesh_basename}")
mesh_path = os.path.join(DATA_MESH_DIR, mesh_basename)
input_mesh, mesh_path = uncert.load_mesh(mesh_path) # This converts PLY to VTK if necessary

# --- Setup shape completion output directory ---
mesh_name = os.path.splitext(os.path.basename(mesh_basename))[0]
OUTPUT_MESH_DIR = os.path.join(OUTPUT_BASE_DIR, mesh_name)
os.makedirs(OUTPUT_MESH_DIR, exist_ok=True)
print(f"Shape completion results will be saved to: {OUTPUT_MESH_DIR}")

# --- Load corresponding original whole mesh ---
if whole_mesh_available:
    # Find path to whole mesh
    wholemesh_path = os.path.join(DATA_WHOLEMESH_DIR, mesh_basename.replace("_partial", ""))
    if not os.path.exists(wholemesh_path):
        if os.path.splitext(wholemesh_path)[1] == '.ply':
            wholemesh_path_alt = wholemesh_path.replace('.ply', '.vtk')
        elif os.path.splitext(wholemesh_path)[1] == '.vtk':
            wholemesh_path_alt = wholemesh_path.replace('.vtk', '.ply')
        if os.path.exists(wholemesh_path_alt):
            wholemesh_path = wholemesh_path_alt
        else:
            print("Warning: Cannot find corresponding whole mesh for the given partial mesh. Proceeding without ground truth whole mesh.")
            whole_mesh_available = False
if whole_mesh_available:
    # Convert PLY to VTK
    whole_mesh, wholemesh_path = uncert.load_mesh(wholemesh_path)
    # Extract name and basename
    wholemesh_basename = os.path.basename(wholemesh_path)
    wholemesh_name = os.path.splitext(wholemesh_basename)[0]

In [ ]:
# --- Preview selected mesh ---
mesh_list = [input_mesh]
if whole_mesh_available:
    mesh_list.append(whole_mesh)
preview_meshes(mesh_list, jupyter_backend='html')

## 4. Shape completion: Latent optimization and mesh reconstruction
---
Optimize latents of input partial mesh and reconstruct mesh.

### Optimize latents

In [ ]:
# --- Optimize latent code, or load existing ---
use_latent_from_file = False

optimal_latent_filename = f"{mesh_name}_optimal_latent.pth"
optimal_latent_path = os.path.join(OUTPUT_MESH_DIR, optimal_latent_filename)
if use_latent_from_file and optimal_latent_filename in os.listdir(OUTPUT_MESH_DIR):
    # Load existing optimal latent
    optimal_latent = torch.load(optimal_latent_path)
    print(f"Existing optimal latent code for {mesh_name} loaded from:")
    print(optimal_latent_path)
else:
    # Optimize latent code and save
    optimal_latent = uncert.compute_optimal_latent(mesh_path, model, config, latent_codes, l2_loss=False, n_samples=1024, device=device)
    torch.save(optimal_latent, optimal_latent_path)
    print(f"Optimal latent code for {mesh_name} saved to:")
    print(optimal_latent_path)

### Reconstruct mesh from optimal latent code

In [ ]:
# --- Reconstruct mesh from optimal latents, or load existing ---
use_recon_mesh_from_file = False

recon_mesh_filename = f"{mesh_name}_shape_completion.vtk"
recon_mesh_path = os.path.join(OUTPUT_MESH_DIR, recon_mesh_filename)
if use_recon_mesh_from_file and recon_mesh_filename in os.listdir(OUTPUT_MESH_DIR):
    # Load existing reconstructed mesh
    recon_mesh = pv.read(recon_mesh_path)
    print(f"Existing reconstructed mesh for {mesh_name} loaded from:")
    print(recon_mesh_path)
else:
    # Reconstruct mesh from optimal latent code and save
    recon_mesh = uncert.reconstruct_mesh(optimal_latent, model, device, verbose=True)
    recon_mesh.save(recon_mesh_path)
    print(f"Reconstructed mesh (completed shape) for {mesh_name} saved to:")
    print(recon_mesh_path)

In [ ]:
# --- Print mesh properties ---
print("Input mesh info:\n", input_mesh)
print("Reconstructed mesh info:\n", recon_mesh)
if whole_mesh_available:
    print("Whole mesh info:\n", whole_mesh)
    chamfer_dist = uncert.compute_chamfer(uncert.unit_scale_mesh(recon_mesh).points, uncert.unit_scale_mesh(whole_mesh).points)
    print(f"Chamfer distance: {chamfer_dist}")
    #0.17 ish

# --- Preview shape completion result ---
mesh_list = [uncert.unit_scale_mesh(input_mesh), recon_mesh]
if whole_mesh_available:
    mesh_list.append(uncert.unit_scale_mesh(whole_mesh))
preview_meshes(mesh_list, jupyter_backend='html')

## 5. Uncertainty estimation: Hyperparameter tuning (optional)
---
**If a ground truth whole mesh is available**, it can be used to tune the parameters $\sigma_Y$ and $\sigma_z$ used during Laplace approximation of the posterior probability distribution of the optimal latent code, resulting in a more accurate uncertainty estimation result.

In [ ]:
# --- Measure NLL of the predicted SDF distribution at different configurations ---

# Tuning output directory and CSV output path
OUTPUT_UNCTUNING_DIR = os.path.join(OUTPUT_MESH_DIR, "uncertainty_tuning")
os.makedirs(OUTPUT_UNCTUNING_DIR, exist_ok=True)
dt_str = datetime.today().strftime("%Y%m%d_%H%M%S")
results_csv_path = os.path.join(OUTPUT_UNCTUNING_DIR, f"results_{dt_str}.csv")

# Load reconstructed mesh and ground truth whole mesh
recon_mesh_dec = uncert.decimate_mesh(recon_mesh, n_triangles=5000)
preds = decode_sdf(model, optimal_latent, torch.tensor(recon_mesh_dec.points).to(device)).detach().cpu().numpy().flatten()

# Instantiate and reuse approximators (model and optimal_latent do not change)
laplace = uncert.LaplaceApproximator(model, optimal_latent, mesh_path, config, device)
montecarlo = uncert.MonteCarloUncertainty(model, optimal_latent, device)
analytical = uncert.AnalyticalUncertainty(model, optimal_latent, device)

# Parameter values to evaluate
data_stds = np.concat([[0],np.logspace(-6,0,16)])
latent_stds = np.concat([[0],np.logspace(-6,0,16)])
prop_types = ['analytical'] # 'montecarlo' can also be included, but is much slower.
n_combinations = len(list(itertools.product(data_stds, latent_stds, prop_types)))

# Rank parameter sets by NLL
results_log = []
best_idx = 0
for i, (dstd, lstd, prop_type) in enumerate(itertools.product(data_stds, latent_stds, prop_types)):
    print(f"Parameter set {i+1}/{n_combinations}: {dstd=}, {lstd=}, {prop_type=}")
    # Approximate covariance of latent posterior
    try:
        covariance, hessian = laplace.covariance(dstd, lstd, data_weight=int(dstd!=0), latent_weight=int(lstd!=0), n_samples=2000, recompute_jacobian=False)
    except:
        print("Could not construct covariance")
        continue
    # Propagate latent covariance to SDF uncertainty at each recon mesh vertex
    if prop_type == 'analytical':
        # Local linearization analysis
        surface_sdf_std = analytical.sdf_uncertainty(covariance, recon_mesh_dec.points, recompute_gradients=False)
    elif prop_type == 'montecarlo':
        # Monte Carlo sampling
        try:
            surface_sdf_std = montecarlo.sdf_uncertainty(covariance, recon_mesh_dec.points, n_samples=400)
        except:
            print("Could not construct covariance")
            continue
    else:
        raise ValueError("Invalid uncertainty propagation type.")
    # NLL and TV
    nll = uncert.eval_nll(preds, surface_sdf_std.detach().cpu().numpy(), uncert.unit_scale_mesh(recon_mesh_dec).points, uncert.unit_scale_mesh(whole_mesh))
    tv = uncert.eval_tv((surface_sdf_std**2).detach().cpu().numpy())
    # Save results
    results_log.append(dict(dstd=dstd, lstd=lstd, prop_type=prop_type, nll=nll, tv=tv))
    print(results_log[-1])
    if results_log[-1]['nll'] < results_log[best_idx]['nll']:
        best_idx = len(results_log) - 1
    pd.DataFrame(results_log).to_csv(results_csv_path)

print("Best results:", results_log[best_idx])
print(f"All results saved to {results_csv_path}")

In [ ]:
# --- Plot and save the NLL values of each configuration tested ---

results_df = pd.DataFrame(results_log)
for prop_type in prop_types:
    # Single figure with broken axes
    fig, ((ax_y,ax_xy),(ax,ax_x)) = plt.subplots(2,2, width_ratios=(1.0,0.1), height_ratios=(0.1,1.0), figsize=(6.0,6.0))
    fig.subplots_adjust(hspace=0.03,wspace=0.03)
    # Main (bottom left)
    ax.spines.top.set_visible(False)
    ax.spines.right.set_visible(False)
    ax.set_xlabel(r"$\sigma_Y$")
    ax.set_ylabel(r"$\sigma_z$")
    ax.set_xscale('log')
    ax.set_yscale('log')
    # Right strip
    ax_x.spines.left.set_visible(False)
    ax_x.spines.top.set_visible(False)
    ax_x.yaxis.tick_right()
    ax_x.set_yscale('log')
    ax_x.set_xlim(-1e-12, 1e-12)
    ax_x.set_xticks([0])
    ax_x.set_xticklabels([r"$\infty$"])
    # Upper strip
    ax_y.spines.bottom.set_visible(False)
    ax_y.spines.right.set_visible(False)
    ax_y.xaxis.tick_top()
    ax_y.set_xscale('log')
    ax_y.set_ylim(-1e-12, 1e-12)
    ax_y.set_yticks([0])
    ax_y.set_yticklabels([r"$\infty$"])
    # Upper-right corner
    ax_xy.spines.left.set_visible(False)
    ax_xy.spines.bottom.set_visible(False)
    ax_xy.xaxis.tick_top()
    ax_xy.yaxis.tick_right()
    ax_xy.set_xlim(-1e-12, 1e-12)
    ax_xy.set_ylim(-1e-12, 1e-12)
    ax_xy.set_xticks([0])
    ax_xy.set_yticks([0])
    ax_xy.set_xticklabels([r"$\infty$"])
    ax_xy.set_yticklabels([r"$\infty$"])
    # Plot data as points
    mask = results_df['prop_type']==prop_type
    pdfs = np.exp(-results_df[mask]['nll']).astype(float)
    idx_best = np.argmax(pdfs)
    edgecolors = ['none']*len(pdfs)
    edgecolors[idx_best] = 'red'
    sizes = np.full_like(pdfs, 8**2)
    scat = ax.scatter(results_df[mask]['dstd'], results_df[mask]['lstd'], c=pdfs, edgecolors=edgecolors, s=sizes)
    scat_x = ax_x.scatter(results_df[mask]['dstd'], results_df[mask]['lstd'], c=pdfs, edgecolors=edgecolors, s=sizes)
    scat_y = ax_y.scatter(results_df[mask]['dstd'], results_df[mask]['lstd'], c=pdfs, edgecolors=edgecolors, s=sizes)
    scat_xy = ax_xy.scatter(results_df[mask]['dstd'], results_df[mask]['lstd'], c=pdfs, edgecolors=edgecolors, s=sizes)
    cax = fig.add_axes((1.0,0.2,0.03,0.6))
    fig.colorbar(scat_x, cax=cax, label=r"Average PDF, $p_\theta(y | x, z)$")
    fig.suptitle(f"Average probability density of GT SDF wrt predicted SDF distribution\nvia Laplace approximation, {prop_type=}\n{mesh_name}", y=1.05)
    results_fig_path = os.path.join(OUTPUT_UNCTUNING_DIR, f"results_{dt_str}_{prop_type}.png")
    fig.savefig(results_fig_path)
    fig.show()
    print(f"Plot saved to {results_fig_path}")

## 6. Uncertainty estimation: Laplace approximation and covariance propagation
---
First, use Laplace approximation to compute the posterior of the optimal latent code as a Gaussian distribution. Then, propagate the covariance through the model to compute SDF variance, or SDF uncertainty.

### Choose hyperparameters
Choose appropriate values for `data_std` $\sigma_Y$ and `latent_std` $\sigma_z$ for use in Laplace approximation.
- $\sigma_Y$ is the expected data (SDF) standard deviation. 
    - This can be interpreted as the expected error or noise between the model's SDF predictions and the ground truth SDF values. It reflects confidence in the model to reconstruct the correct ground truth SDF given the input partial mesh.
- $\sigma_z$ is the expected latent standard deviation.
    - This can be interpreted as the expected error or noise between the computed optimal latents and the true optimal latents for the ground truth shape. It reflects confidence in the model's encoding of the input partial mesh.
- **If uncertainty hyperparameter tuning was run with a ground truth whole mesh, use the best parameter values found.**
- **If choosing values manually (for test data), consider values approximately between `1e-5` and `1e0`.**
    - Values around $10^{-5}$ yield the best results for the case where the input mesh is the ground truth whole mesh itself. Therefore, this should be for cases where you are extremely confident that the model can reconstruct the correct shape.
    - The suggested upper limit is $10^{0}$ because both mesh positions and latents occur approximately within the range $[-1,1]$. This standard deviation spans the entire domain and corresponds to cases where confidence in shape completion accuracy is lowest.
- A `latent_weight` of 0 is equivalent to infinite latent variance, or a uniform latent prior.
- A `data_weight` of 0 is equivalent to infinite SDF variance, ignoring reconstruction and assuming a spherical latent posterior.

In [ ]:
data_std = 6e-2 # Expected data std dev (noise), based on quality of input mesh
latent_std = 4e-1 # Expected latent std dev (noise), based on stability of optimal latent
data_weight = 1 # Weight of reconstruction (data) term
latent_weight = 1 # Weight of regularization (latent) term

### Initialize approximator and propagators

In [ ]:
# --- Laplace approximator ---
laplace = uncert.LaplaceApproximator(model, optimal_latent, mesh_path, config, device, verbose=True)

# --- Uncertainty propagator ---
propagation_mode = 'montecarlo' # Select 'montecarlo' or 'analytical'
if propagation_mode == 'analytical':
    propagator = uncert.AnalyticalUncertainty(model, optimal_latent, device, verbose=True)
elif propagation_mode == 'montecarlo':
    propagator = uncert.MonteCarloUncertainty(model, optimal_latent, device, verbose=True)

### Approximate latent posterior

In [ ]:
# --- Compute approximate covariance of latent posterior ---
covariance, hessian = laplace.covariance(data_std, latent_std, data_weight, latent_weight, n_samples=2000, recompute_jacobian=False)

### Propagate latent covariance to get SDF uncertainty

In [ ]:
# --- Compute SDF uncertainty at mesh vertices ---
recon_mesh_dec = uncert.decimate_mesh(recon_mesh, n_triangles=5000, verbose=True) # Reduce vertex count to speed up computation
if propagation_mode == 'analytical':
    surface_sdf_std = analytical.sdf_uncertainty(covariance, recon_mesh_dec.points, recompute_gradients=False)
elif propagation_mode == 'montecarlo':
    surface_sdf_std = montecarlo.sdf_uncertainty(covariance, recon_mesh_dec.points, n_samples=2000)

# --- Update mesh point data and save ---
# Create output directory for current hyperparameters
OUTPUT_UNC_DIR = os.path.join(OUTPUT_MESH_DIR, f"data{data_std if data_weight else 'inf'}_lat{latent_std if latent_weight else 'inf'}")
os.makedirs(OUTPUT_UNC_DIR, exist_ok=True)
# Update mesh data
recon_mesh_dec.point_data['SdfUncertainty'] = surface_sdf_std.cpu().numpy()
recon_mesh_dec_unc_path = os.path.join(OUTPUT_UNC_DIR, f"{mesh_name}_shape_completion_unc.vtk")
recon_mesh_dec.save(recon_mesh_dec_unc_path)
print("Reconstructed decimated mesh with uncertainty saved to:")
print(recon_mesh_dec_unc_path)
# Print mesh properties
print(f"Mesh info: {recon_mesh_dec}")
print(f"Mesh point data: {recon_mesh_dec.point_data}")

## 7. Visualize results
---
Plot the completed shape and visualize the SDF uncertainty at each point on the surface. When inspecting the result, consider both the magnitude and distribution of the uncertainty values, which refer to the standard deviation of the predicted SDF value at each surface point.

In [ ]:
# --- Plot and save png, gif ---
out_name = os.path.join(OUTPUT_UNC_DIR, f"{mesh_name}_recon_mesh_unc_an")
render_and_save_uncertainty(input_mesh, recon_mesh_dec, whole_mesh if whole_mesh_available else None, 'SdfUncertainty', mesh_name, wholemesh_basename, out_name, 'html')